# Music Annotation Project
## Notebook 72: Frozen V8.8 Annotated-Only Supervised Operational Trial

**Runtime:** CPU by default. Use GPU only if the resolved frozen V8.8 adapter requires it.

This notebook processes every eligible reviewed page from `source/annotated_only` and `source/annotated` through one identical clean-free runtime path. It locks inputs, decisions, probabilities and corrected pages before resolving `source/clean`. Clean truth is used only after the prediction lock for paired-page scoring.

### Fixed contract
- reviewer-corrected authoritative mask is the sole intervention authority;
- no `source/clean` access before the prediction lock is CLOSED;
- frozen V8.8 only: no training, refitting, recalibration, threshold movement or policy change;
- zero changed pixels outside the prepared authoritative mask;
- exact frozen fallback behaviour outside valid V8.8 interventions;
- every candidate page is processed, excluded or failed with an explicit reason;
- every processed page receives a blinded original-versus-corrected usability package;
- no automatic production or mask-authority release.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow gradio pillow scikit-image

## 0. Configuration and immutable run directory

For the definitive run, pin `V88_OPERATOR_PATH` and `V88_BUNDLE_ROOT` to the exact frozen release. The preferred adapter is a durable `v8_8_operator.py` exposing `load_operator(...)` and `run_page(observed, authoritative_seed_mask, runtime_context)`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
import builtins, contextlib, hashlib, importlib.util, json, os, platform, random, shutil, sys, time, traceback, zipfile
import cv2, numpy as np, pandas as pd

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
ANNOTATED_ONLY_DIR=ROOT/'source'/'annotated_only'
ANNOTATED_DIR=ROOT/'source'/'annotated'
# Deliberately a string before lock. Do not instantiate Path or enumerate it pre-lock.
CLEAN_DIR_LITERAL='/content/drive/MyDrive/MusicAnnotationProject/source/clean'
HITL_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
HITL_DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEWED_MASK_FALLBACK_DIRS=[ROOT/'production_pilot'/'reviewed_masks', ROOT/'human_in_the_middle'/'corrected_masks']
OUTPUT_ROOT=ROOT/'notebook72_runs'

# Exact frozen identities. Set overrides for the definitive run.
V88_OPERATOR_PATH=ROOT/'v8_8_release'/'v8_8_operator.py'
V88_BUNDLE_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'/'20261102_123709_notebook52_v8_8a'
ACTIVE_RELEASE_POINTER=None #ROOT/'v8_8_registered_replay_release_control/runs/20261109_125704_notebook57R1_v8_8f'/'release_decision.json'

SMOKE_TEST=False
SMOKE_TEST_PAGES=5
PAGE_ALLOWLIST=None
PUBLIC_LINK=True
RANDOM_SEED=72
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
EXPECTED_API_VERSION='notebook72-v1'

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72_frozen_v8_8_operational_trial'
OUT=OUTPUT_ROOT/RUN_ID
D00=OUT/'00_configuration'; D01=OUT/'01_operator_verification'; D02=OUT/'02_roster_and_lineage'
D03=OUT/'03_runtime_outputs'; D04=OUT/'04_runtime_integrity'; D05=OUT/'05_prediction_lock'
D06=OUT/'06_paired_truth_evaluation'; D07=OUT/'07_blinded_review'; D08=OUT/'08_combined_results'; D09=OUT/'09_replay'; UPLOAD=OUT/'upload'
for p in [OUT,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Notebook 72 output:',OUT)

## 1. Core helpers, access audit and atomic output controls

In [92]:
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    path=Path(path); require(path.exists() and path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),default=str).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text())
def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def imread(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(im is not None,f'Unreadable image: {path}'); return im

def semantic_npz_hash(path):
    with np.load(path,allow_pickle=False) as z: return object_hash({k:array_hash(z[k]) for k in sorted(z.files)})

def directory_inventory(root):
    rows=[]
    for p in sorted(Path(root).rglob('*')):
        if p.is_file(): rows.append({'relative_path':str(p.relative_to(root)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    return pd.DataFrame(rows)

ACCESS_ROWS=[]
LOCK_STATE='OPEN'
_original_open=builtins.open

def audited_open(file,*args,**kwargs):
    raw=str(file)
    clean_hit='/source/clean/' in raw.replace('\\','/') or raw.replace('\\','/').endswith('/source/clean')
    ACCESS_ROWS.append({'utc':datetime.now(timezone.utc).isoformat(),'lock_state':LOCK_STATE,'path':raw,'clean_path':clean_hit,'mode':str(args[0] if args else kwargs.get('mode','r'))})
    if LOCK_STATE!='CLOSED' and clean_hit: raise RuntimeError(f'TRUTH FIREWALL: clean access before lock: {raw}')
    return _original_open(file,*args,**kwargs)

@contextlib.contextmanager
def truth_firewall():
    builtins.open=audited_open
    try: yield
    finally: builtins.open=_original_open

## 2. Resolve and verify the frozen V8.8 operator bundle

The notebook fails closed rather than recreating V8.8 from historical fragments. A compatible adapter must return a page result containing `corrected`, `baseline`, `prepared_mask`, `decisions`, `probabilities`, `changed`, and `audit`.

In [ ]:
def resolve_active_bundle():
    global V88_OPERATOR_PATH,V88_BUNDLE_ROOT
    if V88_BUNDLE_ROOT is None and ACTIVE_RELEASE_POINTER.exists():
        active=read_json(ACTIVE_RELEASE_POINTER)
        manifest=Path(active.get('release_manifest',''))
        if manifest.exists(): V88_BUNDLE_ROOT=manifest.parent
    require(V88_BUNDLE_ROOT is not None,'Set V88_BUNDLE_ROOT or provide an active V8.8 release pointer.')
    V88_BUNDLE_ROOT=Path(V88_BUNDLE_ROOT)
    require(V88_BUNDLE_ROOT.exists(),f'V8.8 bundle root missing: {V88_BUNDLE_ROOT}')
    if V88_OPERATOR_PATH is None:
        candidates=list(V88_BUNDLE_ROOT.rglob('v8_8_operator.py'))+list(ROOT.rglob('v8_8_operator.py'))
        candidates=list({str(p.resolve()):p.resolve() for p in candidates}.values())
        require(len(candidates)==1,f'Expected one v8_8_operator.py; found {candidates}')
        V88_OPERATOR_PATH=candidates[0]
    V88_OPERATOR_PATH=Path(V88_OPERATOR_PATH)
    require(V88_OPERATOR_PATH.exists(),f'Operator missing: {V88_OPERATOR_PATH}')

resolve_active_bundle()
spec=importlib.util.spec_from_file_location('v8_8_operator',V88_OPERATOR_PATH)
v88_module=importlib.util.module_from_spec(spec);
sys.modules['v8_8_operator'] = v88_module
spec.loader.exec_module(v88_module)
require(hasattr(v88_module,'load_operator') and hasattr(v88_module,'run_page'),'Adapter must expose load_operator and run_page.')
operator=v88_module.load_operator(V88_BUNDLE_ROOT)
operator_identity={
 'api_version':getattr(v88_module,'API_VERSION',None),
 'operator_path':str(V88_OPERATOR_PATH),'operator_sha256':sha256_file(V88_OPERATOR_PATH),
 'bundle_root':str(V88_BUNDLE_ROOT),'bundle_inventory':directory_inventory(V88_BUNDLE_ROOT).to_dict('records')
}
require(operator_identity['api_version'] in [None,EXPECTED_API_VERSION],f'Incompatible adapter API: {operator_identity["api_version"]}')
operator_identity['bundle_inventory_sha256']=object_hash(operator_identity['bundle_inventory'])
atomic_json(D01/'operator_identity.json',operator_identity)
pd.DataFrame(operator_identity['bundle_inventory']).to_csv(D01/'operator_artifact_inventory.csv',index=False)
print('Operator:',V88_OPERATOR_PATH); print('Bundle:',V88_BUNDLE_ROOT)

## 3. Frozen operator identity and load gate

In [ ]:
# ============================================================
# 3. Frozen replay gate
# ============================================================

require(
    hasattr(v88_module, 'load_operator'),
    'Adapter must expose load_operator(v88_bundle_root).'
)

require(
    hasattr(v88_module, 'run_page'),
    'Adapter must expose run_page(observed, authoritative_seed_mask, runtime_context).'
)

# load_operator() is the frozen identity/integrity gate.
# It validates the completed Notebook 52 freeze and resolves the
# authoritative Notebook 50 run automatically.
try:
    operator = v88_module.load_operator(V88_BUNDLE_ROOT)

    replay = {
        'passed': True,
        'gate': 'frozen_operator_load_and_identity',
        'operator_id': getattr(v88_module, 'OPERATOR_ID', None),
        'operator_version': getattr(v88_module, 'OPERATOR_VERSION', None),
        'v88_bundle_root': str(operator.v88_bundle_root),
        'notebook50_run': str(operator.notebook50_run),
        'policy_id': operator.policy_id,
        'policy_manifest': str(operator.policy_manifest_path),
        'predictor_schema': str(operator.predictor_schema_path),
        'registered_decision_table': str(operator.decision_table_path),
        'metadata': dict(operator.metadata),
    }

except Exception as exc:
    replay = {
        'passed': False,
        'gate': 'frozen_operator_load_and_identity',
        'v88_bundle_root': str(V88_BUNDLE_ROOT),
        'error_type': type(exc).__name__,
        'error': str(exc),
    }

atomic_json(D09 / 'replay_test.json', replay)

require(
    replay.get('passed') is True,
    f'Frozen V8.8 replay/identity gate failed: {replay}'
)

print(json.dumps(replay, indent=2, default=str))

## 4. Build the full runtime roster and resolve authoritative review lineage

Resolution follows the hash-based HITL pattern used by the supervised predecessor notebooks. Reviewer audit records take priority. There is no fallback to unreviewed draft masks.

In [ ]:
# ============================================================
# Build reviewed-mask runtime roster
# ============================================================

def md5_file(path):
    path = Path(path)

    require(
        path.is_file(),
        f"Cannot calculate MD5; file does not exist: {path}"
    )

    h = hashlib.md5()

    with _original_open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def load_audits(folder):
    rows = []
    failures = []

    folder = Path(folder)

    if not folder.exists():
        return rows, failures

    for path in sorted(folder.glob("*.json")):
        try:
            value = json.loads(path.read_text())

            require(
                isinstance(value, dict),
                f"Non-object audit: {path}"
            )

            value["_audit_path"] = str(path)
            rows.append(value)

        except Exception as exc:
            failures.append({
                "audit_path": str(path),
                "error_type": type(exc).__name__,
                "error": str(exc),
            })

    return rows, failures


review_audits, parse_failures = load_audits(HITL_AUDIT_DIR)

parse_failure_columns = [
    "audit_path",
    "error_type",
    "error",
]

pd.DataFrame(
    parse_failures,
    columns=parse_failure_columns,
).to_csv(
    D02 / "audit_parse_failures.csv",
    index=False,
)

require(
    not parse_failures,
    "Unreadable review audit records exist. "
    f"See {D02 / 'audit_parse_failures.csv'}"
)


def _normalise_path_text(value):
    """
    Return a usable path string or None.

    This check must happen before Path(value), because Path("")
    silently becomes Path("."), which caused the prior error.
    """
    if value is None:
        return None

    text = str(value).strip()

    if text in {"", ".", "./"}:
        return None

    return text


def _same_path(left, right):
    """
    Best-effort exact path comparison for older audit schemas.
    This is only a secondary lookup after source_md5.
    """
    left = _normalise_path_text(left)
    right = _normalise_path_text(right)

    if left is None or right is None:
        return False

    try:
        return Path(left).expanduser().resolve() == Path(right).expanduser().resolve()
    except Exception:
        return left == right


def _lineage_exclusion(source, digest, reason, **details):
    """
    Standard result for a source that is not eligible for the
    reviewed-mask V8.8 runtime trial.
    """
    return None, {
        "reason": reason,
        "source_md5": digest,
        **details,
    }


def find_review_lineage(source):
    source = Path(source)
    digest = md5_file(source)

    # --------------------------------------------------------
    # Primary lookup: exact source-content identity
    # --------------------------------------------------------
    matches = [
        record
        for record in review_audits
        if str(record.get("source_md5", "")).strip().lower()
        == digest.lower()
    ]

    match_method = "source_md5"

    # --------------------------------------------------------
    # Secondary lookup: exact path for older audit schemas
    # --------------------------------------------------------
    if not matches:
        matches = [
            record
            for record in review_audits
            if (
                _same_path(record.get("source"), source)
                or _same_path(record.get("source_file"), source)
            )
        ]

        match_method = "legacy_exact_source_path"

    # Duplicate authoritative records are an integrity problem.
    require(
        len(matches) <= 1,
        f"{source.name}: duplicate review audits: "
        f"{[record.get('_audit_path') for record in matches]}"
    )

    # A source page without a completed review is expected to be
    # excluded from this reviewed-mask trial.
    if not matches:
        return _lineage_exclusion(
            source,
            digest,
            "NO_REVIEW_AUDIT",
            match_method=None,
            review_audit_path=None,
            observed_path=None,
            seed_mask_path=None,
        )

    review = matches[0]
    audit_path = Path(review["_audit_path"])

    # Do not fall back to source here. A completed reviewed lineage
    # must explicitly identify its aligned output.
    observed_raw = _normalise_path_text(
        review.get("aligned_output")
    )

    if observed_raw is None:
        return _lineage_exclusion(
            source,
            digest,
            "REVIEW_AUDIT_MISSING_ALIGNED_OUTPUT",
            match_method=match_method,
            review_audit_path=str(audit_path),
            observed_path=None,
            seed_mask_path=None,
        )

    # Do not call Path() until after checking for missing/blank values.
    mask_raw = _normalise_path_text(
        review.get("corrected_mask_output")
    )

    if mask_raw is None:
        return _lineage_exclusion(
            source,
            digest,
            "REVIEW_AUDIT_MISSING_CORRECTED_MASK_OUTPUT",
            match_method=match_method,
            review_audit_path=str(audit_path),
            observed_path=observed_raw,
            seed_mask_path=None,
        )

    observed_path = Path(observed_raw).expanduser()
    seed_mask_path = Path(mask_raw).expanduser()

    if not observed_path.is_file():
        return _lineage_exclusion(
            source,
            digest,
            "REVIEWED_OBSERVED_MISSING",
            match_method=match_method,
            review_audit_path=str(audit_path),
            observed_path=str(observed_path),
            seed_mask_path=str(seed_mask_path),
        )

    if not seed_mask_path.is_file():
        return _lineage_exclusion(
            source,
            digest,
            "AUTHORITATIVE_MASK_MISSING",
            match_method=match_method,
            review_audit_path=str(audit_path),
            observed_path=str(observed_path),
            seed_mask_path=str(seed_mask_path),
        )

    # Validate that an audit source path, if still present, has not
    # silently changed content.
    audit_source_raw = (
        _normalise_path_text(review.get("source"))
        or _normalise_path_text(review.get("source_file"))
    )

    if audit_source_raw is not None:
        audit_source_path = Path(audit_source_raw).expanduser()

        if audit_source_path.is_file():
            audit_source_md5 = md5_file(audit_source_path)

            if audit_source_md5.lower() != digest.lower():
                return _lineage_exclusion(
                    source,
                    digest,
                    "REVIEW_AUDIT_SOURCE_HASH_STALE",
                    match_method=match_method,
                    review_audit_path=str(audit_path),
                    audit_source_path=str(audit_source_path),
                    audit_source_md5=audit_source_md5,
                    observed_path=str(observed_path),
                    seed_mask_path=str(seed_mask_path),
                )

    return {
        "observed_path": observed_path,
        "seed_mask_path": seed_mask_path,
        "review_audit_path": audit_path,
        "source_md5": digest,
        "review_match_method": match_method,
    }, None


# ============================================================
# Enumerate both cohorts
# ============================================================

sources = []

for cohort, folder in [
    ("A_ANNOTATED_ONLY", ANNOTATED_ONLY_DIR),
    ("B_PAIRED_AS_ANNOTATED_ONLY", ANNOTATED_DIR),
]:
    for stem, path in image_map(folder).items():
        sources.append({
            "page": str(stem),
            "cohort": cohort,
            "source_path": Path(path),
        })


# ============================================================
# Resolve authoritative review lineage
# ============================================================

seen_hash = {}
roster_rows = []
exclusion_rows = []

for item in sorted(
    sources,
    key=lambda row: (
        row["page"],
        row["cohort"],
        str(row["source_path"]),
    ),
):
    path = Path(item["source_path"])
    source_sha256 = sha256_file(path)

    # --------------------------------------------------------
    # Exact duplicate-content exclusion
    # --------------------------------------------------------
    if source_sha256 in seen_hash:
        exclusion_rows.append({
            **item,
            "source_path": str(path),
            "source_sha256": source_sha256,
            "status": "EXCLUDED",
            "reason": "DUPLICATE_CONTENT",
            "duplicate_of": seen_hash[source_sha256],
            "source_md5": None,
            "review_audit_path": None,
            "observed_path": None,
            "seed_mask_path": None,
        })
        continue

    seen_hash[source_sha256] = item["page"]

    # --------------------------------------------------------
    # Review-lineage lookup
    # --------------------------------------------------------
    lineage, failure = find_review_lineage(path)

    if failure is not None:
        exclusion_rows.append({
            **item,
            "source_path": str(path),
            "source_sha256": source_sha256,
            "status": "EXCLUDED",
            **failure,
        })
        continue

    # --------------------------------------------------------
    # Images are read only after valid concrete files resolve
    # --------------------------------------------------------
    try:
        observed = imread(lineage["observed_path"])
    except Exception as exc:
        exclusion_rows.append({
            **item,
            "source_path": str(path),
            "source_sha256": source_sha256,
            "source_md5": lineage["source_md5"],
            "review_audit_path": str(lineage["review_audit_path"]),
            "observed_path": str(lineage["observed_path"]),
            "seed_mask_path": str(lineage["seed_mask_path"]),
            "review_match_method": lineage["review_match_method"],
            "status": "EXCLUDED",
            "reason": "REVIEWED_OBSERVED_UNREADABLE",
            "error_type": type(exc).__name__,
             "error": str(exc),
        })
        continue

    try:
        seed_mask = imread(lineage["seed_mask_path"])
    except Exception as exc:
        exclusion_rows.append({
            **item,
            "source_path": str(path),
            "source_sha256": source_sha256,
            "source_md5": lineage["source_md5"],
            "review_audit_path": str(lineage["review_audit_path"]),
            "observed_path": str(lineage["observed_path"]),
            "seed_mask_path": str(lineage["seed_mask_path"]),
            "review_match_method": lineage["review_match_method"],
            "status": "EXCLUDED",
           "reason": "AUTHORITATIVE_MASK_UNREADABLE",
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        continue

    # --------------------------------------------------------
    # No resizing or opportunistic repair
    # --------------------------------------------------------
    if observed.shape != seed_mask.shape:
        exclusion_rows.append({
            **item,
            "source_path": str(path),
            "source_sha256": source_sha256,
            "source_md5": lineage["source_md5"],
            "review_audit_path": str(lineage["review_audit_path"]),
            "observed_path": str(lineage["observed_path"]),
            "seed_mask_path": str(lineage["seed_mask_path"]),
            "review_match_method": lineage["review_match_method"],
            "observed_shape": str(tuple(observed.shape)),
            "seed_mask_shape": str(tuple(seed_mask.shape)),
            "status": "EXCLUDED",
            "reason": "OBSERVED_MASK_SHAPE_MISMATCH",
        })
        continue

    roster_rows.append({
        **item,
        "source_path": str(path),
        "observed_path": str(lineage["observed_path"]),
        "seed_mask_path": str(lineage["seed_mask_path"]),
        "review_audit_path": str(lineage["review_audit_path"]),
        "review_match_method": lineage["review_match_method"],
        "source_md5": lineage["source_md5"],
        "source_sha256": source_sha256,
        "observed_sha256": sha256_file(lineage["observed_path"]),
        "seed_mask_sha256": sha256_file(lineage["seed_mask_path"]),
        "observed_shape": str(tuple(observed.shape)),
        "seed_mask_shape": str(tuple(seed_mask.shape)),
        "status": "ELIGIBLE",
    })


# ============================================================
# Construct stable tables
# ============================================================

roster_columns = [
    "page",
    "cohort",
    "source_path",
    "observed_path",
    "seed_mask_path",
    "review_audit_path",
    "review_match_method",
    "source_md5",
    "source_sha256",
    "observed_sha256",
    "seed_mask_sha256",
    "observed_shape",
    "seed_mask_shape",
    "status",
]

exclusion_columns = [
    "page",
    "cohort",
    "source_path",
    "source_md5",
    "source_sha256",
    "status",
    "reason",
    "duplicate_of",
    "match_method",
    "review_match_method",
    "review_audit_path",
    "observed_path",
    "seed_mask_path",
    "audit_source_path",
    "audit_source_md5",
    "observed_shape",
    "seed_mask_shape",
    "error_type",
    "error",
]

roster = pd.DataFrame(
    roster_rows,
    columns=roster_columns,
)

exclusions = pd.DataFrame(
    exclusion_rows,
).reindex(
    columns=exclusion_columns,
)


# ============================================================
# Integrity checks and optional narrowing
# ============================================================

require(
    len(roster) > 0,
    "No eligible pages with completed authoritative HITL review "
    "were found."
)

require(
    not roster.page.duplicated().any(),
    "Eligible roster contains duplicate page stems across cohorts. "
    "Rename or explicitly resolve them."
)

full_eligible_count = len(roster)

if PAGE_ALLOWLIST is not None:
    allowed = set(map(str, PAGE_ALLOWLIST))
    roster = roster[roster.page.isin(allowed)].copy()

if SMOKE_TEST:
    roster = roster.head(SMOKE_TEST_PAGES).copy()

require(
    len(roster) > 0,
    "No eligible pages remain after PAGE_ALLOWLIST/SMOKE_TEST filtering."
)


# ============================================================
# Write the complete audit before runtime processing
# ============================================================

roster.to_csv(
    D02 / "runtime_roster.csv",
    index=False,
)

exclusions.to_csv(
    D02 / "exclusions.csv",
    index=False,
)

exclusion_summary = (
    exclusions.groupby(
        ["cohort", "reason"],
        dropna=False,
    )
    .size()
    .reset_index(name="pages")
    .sort_values(
        ["cohort", "pages", "reason"],
        ascending=[True, False, True],
    )
)

exclusion_summary.to_csv(
    D02 / "exclusion_summary.csv",
    index=False,
)


# ============================================================
# Console summary
# ============================================================

print("Source pages enumerated:", len(sources))
print("Eligible before allowlist/smoke filtering:", full_eligible_count)
print("Runtime roster pages:", len(roster))
print("Excluded pages:", len(exclusions))

print("\nEligible by cohort:")
print(
    roster.groupby("cohort")
    .size()
    .to_string()
)

if len(exclusions):
    print("\nExclusions by reason:")
    print(
        exclusions["reason"]
        .value_counts(dropna=False)
        .to_string()
    )

    stale_or_broken = exclusions[
        exclusions["reason"].isin({
            "REVIEW_AUDIT_MISSING_ALIGNED_OUTPUT",
            "REVIEW_AUDIT_MISSING_CORRECTED_MASK_OUTPUT",
            "REVIEWED_OBSERVED_MISSING",
            "AUTHORITATIVE_MASK_MISSING",
            "REVIEWED_OBSERVED_UNREADABLE",
            "AUTHORITATIVE_MASK_UNREADABLE",
            "REVIEW_AUDIT_SOURCE_HASH_STALE",
            "OBSERVED_MASK_SHAPE_MISMATCH",
        })
    ]

    print("\nBroken or stale reviewed lineage:", len(stale_or_broken))

print("\nRoster:", D02 / "runtime_roster.csv")
print("Exclusions:", D02 / "exclusions.csv")
print("Exclusion summary:", D02 / "exclusion_summary.csv")

## 5. Runtime context firewall audit

In [ ]:
forbidden_tokens=['clean_path','truth','structural_safe','cosmetic_outcome','clean_sha256']
for c in roster.columns:
    require(not any(t in c.lower() for t in forbidden_tokens),f'Forbidden runtime roster field: {c}')
require('source/clean' not in json.dumps(roster.astype(str).to_dict('records')).replace('\\','/'),'Runtime roster contains clean path.')
atomic_json(D00/'configuration.json',{
 'created_utc':datetime.now(timezone.utc).isoformat(),'run_id':RUN_ID,'smoke_test':SMOKE_TEST,
 'annotated_only_dir':str(ANNOTATED_ONLY_DIR),'annotated_dir':str(ANNOTATED_DIR),
 'clean_dir_not_resolved_pre_lock':True,'operator_identity_sha256':sha256_file(D01/'operator_identity.json'),
 'random_seed':RANDOM_SEED,'promotion_authority':False,'automatic_output_authority':False
})
print('Truth path closed. Runtime roster contains no clean path or truth field.')

## 6. Full frozen inference, per-page integrity and atomic outputs

In [ ]:
# ============================================================
# Pre-6. DIAGNOSTIC ONLY:
#    Resolve Notebook 72 runtime pages against Notebook 50
# ============================================================
#
# This cell does NOT run V8.8 and does NOT alter the roster.
#
# It tests:
#   1. exact page-name identity;
#   2. normalised page-name identity;
#   3. existing page_pair_resolution.csv records;
#   4. HITL audit filename aliases;
#   5. exact file-content identity against historical project images;
#   6. observed-image content identity;
#   7. canvas compatibility with Notebook 50 coordinates.
#
# Outputs:
#   D06/page_identity_diagnostic.csv
#   D06/page_identity_summary.csv
#   D06/notebook50_page_inventory.csv
#   D06/page_pair_resolution_candidates.csv
#   D06/identity_diagnostic.json
#
# No proposed alias is authorised automatically.
# ============================================================

import re
import unicodedata
from collections import defaultdict


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    len(roster) > 0,
    "Runtime roster is empty."
)

require(
    hasattr(operator, "decision_table"),
    "Loaded operator does not expose its Notebook 50 decision table."
)

D06.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalise_page_name(value):
    """
    Conservative filename normalisation for diagnostics only.

    This is not sufficient evidence for operational identity.
    """
    if value is None:
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = Path(text).stem
    text = text.casefold()

    # Make common page-number constructions comparable.
    text = re.sub(r"\bpage[\s._-]*0*(\d+)\b", r"page\1", text)

    # Remove punctuation and spacing.
    text = re.sub(r"[^a-z0-9]+", "", text)

    return text


def safe_sha256(path):
    path = Path(path)

    if not path.is_file():
        return None

    return sha256_file(path)


def safe_md5(path):
    path = Path(path)

    if not path.is_file():
        return None

    return md5_file(path)


def safe_shape(path):
    path = Path(path)

    if not path.is_file():
        return None

    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    return tuple(map(int, image.shape))


def pipe_join(values):
    values = [
        str(value)
        for value in values
        if value not in (None, "")
    ]

    return "|".join(sorted(set(values)))


def existing_image_files(folder):
    folder = Path(folder)

    if not folder.exists():
        return []

    return sorted(
        path
        for path in folder.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in SUPPORTED
        )
    )


def first_existing_column(frame, names):
    for name in names:
        if name in frame.columns:
            return name

    return None


# ------------------------------------------------------------
# Load and validate Notebook 50 identities
# ------------------------------------------------------------

n50 = operator.decision_table.copy()

required_n50_columns = {
    "page",
    "y",
    "x",
    "safe_remove",
}

missing_n50_columns = required_n50_columns - set(n50.columns)

require(
    not missing_n50_columns,
    "Notebook 50 decision table is missing fields: "
    f"{sorted(missing_n50_columns)}"
)

n50["page"] = n50["page"].astype(str)
n50["y"] = pd.to_numeric(n50["y"], errors="raise").astype(int)
n50["x"] = pd.to_numeric(n50["x"], errors="raise").astype(int)

require(
    not n50.duplicated(["page", "y", "x"]).any(),
    "Notebook 50 decision table contains duplicate page/y/x rows."
)

n50_pages = sorted(n50["page"].unique())
n50_page_set = set(n50_pages)

n50_normalised_lookup = defaultdict(list)

for page in n50_pages:
    n50_normalised_lookup[normalise_page_name(page)].append(page)


# Notebook 50 page-level coordinate inventory.
n50_inventory_rows = []

for page, group in n50.groupby("page", sort=True):
    n50_inventory_rows.append({
        "notebook50_page": str(page),
        "normalised_page": normalise_page_name(page),
        "decision_rows": int(len(group)),
        "minimum_y": int(group["y"].min()),
        "maximum_y": int(group["y"].max()),
        "minimum_x": int(group["x"].min()),
        "maximum_x": int(group["x"].max()),
        "safe_remove_rows": int(group["safe_remove"].astype(bool).sum()),
    })

n50_inventory = pd.DataFrame(n50_inventory_rows)

n50_inventory.to_csv(
    D06 / "notebook50_page_inventory.csv",
    index=False,
)


# ------------------------------------------------------------
# Discover existing page-resolution tables
# ------------------------------------------------------------

resolution_paths = sorted(
    {
        path.resolve()
        for path in ROOT.rglob("page_pair_resolution.csv")
        if path.is_file()
    },
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

resolution_frames = []
resolution_parse_failures = []

for resolution_path in resolution_paths:
    try:
        frame = pd.read_csv(resolution_path, dtype=str)
        frame["_resolution_path"] = str(resolution_path)
        resolution_frames.append(frame)

    except Exception as exc:
        resolution_parse_failures.append({
            "resolution_path": str(resolution_path),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })

if resolution_frames:
    resolution_records = pd.concat(
        resolution_frames,
        ignore_index=True,
        sort=False,
    )
else:
    resolution_records = pd.DataFrame()

pd.DataFrame(
    resolution_parse_failures,
    columns=[
        "resolution_path",
        "error_type",
        "error",
    ],
).to_csv(
    D06 / "page_resolution_parse_failures.csv",
    index=False,
)

require(
    not resolution_parse_failures,
    "One or more page_pair_resolution.csv files are unreadable. "
    f"See {D06 / 'page_resolution_parse_failures.csv'}"
)


# ------------------------------------------------------------
# Convert page-resolution records into candidate aliases
# ------------------------------------------------------------

resolution_alias_rows = []

if len(resolution_records):
    # Possible columns representing current/original paths or names.
    current_columns = [
        column
        for column in [
            "source_path",
            "source",
            "source_file",
            "annotated_path",
            "annotated",
            "observed_path",
            "aligned_path",
            "original_path",
            "original_filename",
            "filename",
            "current_page",
            "source_page",
        ]
        if column in resolution_records.columns
    ]

    # Possible columns representing historical/project page IDs.
    historical_columns = [
        column
        for column in [
            "page",
            "resolved_page",
            "canonical_page",
            "historical_page",
            "development_page",
            "notebook50_page",
            "page_id",
        ]
        if column in resolution_records.columns
    ]

    for _, record in resolution_records.iterrows():
        current_values = []

        for column in current_columns:
            value = record.get(column)

            if pd.notna(value) and str(value).strip():
                current_values.append(str(value))

        historical_values = []

        for column in historical_columns:
            value = record.get(column)

            if pd.notna(value) and str(value).strip():
                candidate = Path(str(value)).stem

                if candidate in n50_page_set:
                    historical_values.append(candidate)

        for current_value in current_values:
            for historical_value in historical_values:
                resolution_alias_rows.append({
                    "current_value": current_value,
                    "current_stem": Path(current_value).stem,
                    "current_normalised":
                        normalise_page_name(current_value),
                    "notebook50_page": historical_value,
                    "resolution_path":
                        str(record["_resolution_path"]),
                })

resolution_aliases = pd.DataFrame(
    resolution_alias_rows,
    columns=[
        "current_value",
        "current_stem",
        "current_normalised",
        "notebook50_page",
        "resolution_path",
    ],
)

resolution_aliases.to_csv(
    D06 / "page_pair_resolution_candidates.csv",
    index=False,
)


# ------------------------------------------------------------
# Build HITL audit alias evidence
# ------------------------------------------------------------

audit_alias_rows = []

for audit in review_audits:
    audit_path = audit.get("_audit_path")

    audit_source_values = []

    for key in [
        "source",
        "source_file",
        "aligned_output",
        "corrected_mask_output",
    ]:
        value = audit.get(key)

        if value not in (None, "", "."):
            audit_source_values.append({
                "field": key,
                "value": str(value),
                "stem": Path(str(value)).stem,
            })

    explicit_page_values = []

    for key in [
        "page",
        "page_id",
        "canonical_page",
        "resolved_page",
        "development_page",
        "historical_page",
    ]:
        value = audit.get(key)

        if value not in (None, "", "."):
            explicit_page_values.append({
                "field": key,
                "value": Path(str(value)).stem,
            })

    for source_value in audit_source_values:
        for explicit_page in explicit_page_values:
            if explicit_page["value"] in n50_page_set:
                audit_alias_rows.append({
                    "source_stem": source_value["stem"],
                    "source_normalised":
                        normalise_page_name(source_value["stem"]),
                    "notebook50_page": explicit_page["value"],
                    "audit_path": str(audit_path),
                    "source_field": source_value["field"],
                    "page_field": explicit_page["field"],
                })

audit_aliases = pd.DataFrame(
    audit_alias_rows,
    columns=[
        "source_stem",
        "source_normalised",
        "notebook50_page",
        "audit_path",
        "source_field",
        "page_field",
    ],
)

audit_aliases.to_csv(
    D06 / "hitl_audit_page_alias_candidates.csv",
    index=False,
)


# ------------------------------------------------------------
# Historical image inventory
#
# We inventory likely historical runtime folders.
#
# Exact hashes can prove identical files. A matching shape cannot.
# ------------------------------------------------------------

historical_folders = [
    ROOT / "aligned",
    ROOT / "source" / "annotated",
    ROOT / "source" / "annotated_only",
]

# Include V7.5 cache-status source references if available.
try:
    v75_status_candidates = sorted(
        ROOT.rglob("cache_status.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
except Exception:
    v75_status_candidates = []

historical_image_paths = []

for folder in historical_folders:
    historical_image_paths.extend(
        existing_image_files(folder)
    )

historical_image_paths = list(
    dict.fromkeys(
        Path(path).resolve()
        for path in historical_image_paths
    )
)

historical_rows = []

for index, path in enumerate(historical_image_paths, 1):
    # Limit this inventory to files whose stem is a Notebook 50 page
    # or whose normalised stem could resolve uniquely to one.
    exact_page = path.stem if path.stem in n50_page_set else None

    normalised_hits = n50_normalised_lookup.get(
        normalise_page_name(path.stem),
        [],
    )

    unique_normalised_page = (
        normalised_hits[0]
        if len(normalised_hits) == 1
        else None
    )

    candidate_page = exact_page or unique_normalised_page

    if candidate_page is None:
        continue

    historical_rows.append({
        "notebook50_page": candidate_page,
        "historical_image_path": str(path),
        "historical_image_stem": path.stem,
        "match_basis":
            "EXACT_STEM"
            if exact_page is not None
            else "UNIQUE_NORMALISED_STEM",
        "historical_sha256": safe_sha256(path),
        "historical_md5": safe_md5(path),
        "historical_shape": str(safe_shape(path)),
    })

historical_images = pd.DataFrame(
    historical_rows,
    columns=[
        "notebook50_page",
        "historical_image_path",
        "historical_image_stem",
        "match_basis",
        "historical_sha256",
        "historical_md5",
        "historical_shape",
    ],
)

historical_images.to_csv(
    D06 / "historical_notebook50_image_candidates.csv",
    index=False,
)


# Lookups from physical content identity to Notebook 50 IDs.
historical_source_sha_lookup = defaultdict(list)
historical_source_md5_lookup = defaultdict(list)

for _, record in historical_images.iterrows():
    if pd.notna(record["historical_sha256"]):
        historical_source_sha_lookup[
            str(record["historical_sha256"])
        ].append(str(record["notebook50_page"]))

    if pd.notna(record["historical_md5"]):
        historical_source_md5_lookup[
            str(record["historical_md5"])
        ].append(str(record["notebook50_page"]))


# ------------------------------------------------------------
# Evaluate every Notebook 72 runtime page
# ------------------------------------------------------------

diagnostic_rows = []

for _, row in roster.iterrows():
    runtime_page = str(row["page"])
    source_path = Path(str(row["source_path"]))
    observed_path = Path(str(row["observed_path"]))
    seed_mask_path = Path(str(row["seed_mask_path"]))

    source_sha256 = str(row.get("source_sha256", "")) or safe_sha256(source_path)
    source_md5 = str(row.get("source_md5", "")) or safe_md5(source_path)
    observed_sha256 = (
        str(row.get("observed_sha256", ""))
        or safe_sha256(observed_path)
    )

    observed_shape = safe_shape(observed_path)
    seed_shape = safe_shape(seed_mask_path)

    candidate_evidence = defaultdict(list)

    # --------------------------------------------------------
    # 1. Exact current page stem
    # --------------------------------------------------------
    if runtime_page in n50_page_set:
        candidate_evidence[runtime_page].append(
            "EXACT_RUNTIME_PAGE"
        )

    # --------------------------------------------------------
    # 2. Exact source/observed stems
    # --------------------------------------------------------
    for label, stem in [
        ("EXACT_SOURCE_STEM", source_path.stem),
        ("EXACT_OBSERVED_STEM", observed_path.stem),
        ("EXACT_MASK_STEM", seed_mask_path.stem),
    ]:
        if stem in n50_page_set:
            candidate_evidence[stem].append(label)

       # --------------------------------------------------------
    # 3. Unique normalised-name matches
    # --------------------------------------------------------
    for label, value in [
        ("NORMALISED_RUNTIME_PAGE", runtime_page),
        ("NORMALISED_SOURCE_STEM", source_path.stem),
        ("NORMALISED_OBSERVED_STEM", observed_path.stem),
        ("NORMALISED_MASK_STEM", seed_mask_path.stem),
    \]:
        hits = n50_normalised_lookup.get(
            normalise_page_name(value),
            [],
        )

        if len(hits) == 1:
            candidate_evidence[hits[0]].append(label)

    # --------------------------------------------------------
    # 4. Existing page_pair_resolution.csv
    # --------------------------------------------------------
    if len(resolution_aliases):
        runtime_normalised_values = {
            normalise_page_name(runtime_page),
            normalise_page_name(source_path.stem),
            normalise_page_name(observed_path.stem),
            normalise_page_name(seed_mask_path.stem),
        }
        
        matches = resolution_aliases[
            resolution_aliases["current_normalised"].isin(
                runtime_normalised_values
            )
        ]

        for _, match in matches.iterrows():
            candidate_evidence[
                str(match["notebook50_page"])
            ].append(
                "PAGE_PAIR_RESOLUTION:"
                f"{match['resolution_path']}"
            )

    # --------------------------------------------------------
    # 5. HITL review-audit explicit alias fields
    # --------------------------------------------------------
    if len(audit_aliases):
        runtime_normalised_values = {
            normalise_page_name(runtime_page),
            normalise_page_name(source_path.stem),
            normalise_page_name(observed_path.stem),
            normalise_page_name(seed_mask_path.stem),
        }

        matches = audit_aliases[
            audit_aliases["source_normalised"].isin(
                runtime_normalised_values
            )
        ]

        for _, match in matches.iterrows():
            candidate_evidence[
                str(match["notebook50_page"])
            ].append(
                "HITL_AUDIT_ALIAS:"
                f"{match['audit_path']}"
            )

    # --------------------------------------------------------
    # 6. Exact physical source-content identity
    # --------------------------------------------------------
    if source_sha256:
        for page in historical_source_sha_lookup.get(
            source_sha256,
            [],
        ):
            candidate_evidence[page].append(
                "EXACT_SOURCE_SHA256"
            )

    if source_md5:
        for page in historical_source_md5_lookup.get(
            source_md5,
            [],
        ):
            candidate_evidence[page].append(
                "EXACT_SOURCE_MD5"
            )

    # --------------------------------------------------------
    # Classify candidate strength
    # --------------------------------------------------------
    candidates = sorted(candidate_evidence)
    strong_candidates = []

    for candidate in candidates:
        evidence = candidate_evidence[candidate]

        if any(
            item.startswith(prefix)
            for item in evidence
            for prefix in [
                "EXACT_SOURCE_SHA256",
                "EXACT_SOURCE_MD5",
                "PAGE_PAIR_RESOLUTION:",
                "HITL_AUDIT_ALIAS:",
                "EXACT_RUNTIME_PAGE",
                "EXACT_SOURCE_STEM",
                "EXACT_OBSERVED_STEM",
            ]
        ):
            strong_candidates.append(candidate)

    strong_candidates = sorted(set(strong_candidates))

    # Only a single candidate supported by strong evidence can
    # proceed to canvas checking.
    proposed_n50_page = (
        strong_candidates[0]
        if len(strong_candidates) == 1
        else None
    )

    # --------------------------------------------------------
    # Canvas/coordinate diagnostics
    # --------------------------------------------------------
    coordinate_canvas_ok = None
    coordinate_rows = None
    coordinate_max_y = None
    coordinate_max_x = None
    historical_candidate_paths = []
    historical_candidate_hash_match = None
    historical_candidate_shape_match = None

    if proposed_n50_page is not None:
        q = n50[n50["page"] == proposed_n50_page]

        coordinate_rows = int(len(q))
        coordinate_max_y = int(q["y"].max())
        coordinate_max_x = int(q["x"].max())

        if observed_shape is not None:
            height, width = observed_shape

            coordinate_canvas_ok = bool(
                coordinate_max_y < height
                and coordinate_max_x < width
                and int(q["y"].min()) >= 0
                and int(q["x"].min()) >= 0
            )

        if len(historical_images):
            h = historical_images[
                historical_images["notebook50_page"]
                == proposed_n50_page
            ]

            historical_candidate_paths = (
                h["historical_image_path"]
                .astype(str)
                .tolist()
            )

            hash_matches = []

            for path_text in historical_candidate_paths:
                candidate_path = Path(path_text)
                candidate_sha = safe_sha256(candidate_path)

                if observed_sha256 and candidate_sha:
                    hash_matches.append(
                        observed_sha256 == candidate_sha
                    )

            if hash_matches:
                historical_candidate_hash_match = bool(
                    any(hash_matches)
                )

            shape_matches = []

            for path_text in historical_candidate_paths:
                candidate_shape = safe_shape(
                    Path(path_text)
                )

                if (
                    observed_shape is not None
                    and candidate_shape is not None
                ):
                    shape_matches.append(
                        observed_shape == candidate_shape
                    )

            if shape_matches:
                historical_candidate_shape_match = bool(
                    any(shape_matches)
                )

    # --------------------------------------------------------
    # Final diagnostic classification
    # --------------------------------------------------------
    if len(strong_candidates) > 1:
        diagnostic_status = "AMBIGUOUS_MULTIPLE_STRONG_ALIASES"

    elif proposed_n50_page is None:
        if candidates:
            diagnostic_status = (
                "WEAK_NAME_SIMILARITY_ONLY_NOT_AUTHORISED"
            )
        else:
            diagnostic_status = "NO_NOTEBOOK50_IDENTITY_EVIDENCE"

    elif coordinate_canvas_ok is False:
        diagnostic_status = "ALIAS_FOUND_BUT_COORDINATES_OUTSIDE_CANVAS"

    elif historical_candidate_hash_match is True:
        diagnostic_status = "PROVEN_PIXEL_IDENTICAL_HISTORICAL_IMAGE"

    elif historical_candidate_shape_match is True:
        diagnostic_status = (
            "ALIAS_AND_SHAPE_MATCH_HASH_NOT_PROVEN"
        )

    elif coordinate_canvas_ok is True:
        diagnostic_status = (
            "ALIAS_AND_COORDINATE_CANVAS_COMPATIBLE_HASH_NOT_PROVEN"
        )

    else:
        diagnostic_status = (
            "ALIAS_FOUND_CANVAS_NOT_VERIFIED"
        )

    diagnostic_rows.append({
        "runtime_page": runtime_page,
        "cohort": str(row["cohort"]),
        "source_path": str(source_path),
        "observed_path": str(observed_path),
        "seed_mask_path": str(seed_mask_path),
        "source_md5": source_md5,
        "source_sha256": source_sha256,
        "observed_sha256": observed_sha256,
        "observed_shape": str(observed_shape),
        "seed_mask_shape": str(seed_shape),

        "candidate_notebook50_pages":
            pipe_join(candidates),

        "strong_candidate_notebook50_pages":
            pipe_join(strong_candidates),

        "proposed_notebook50_page":
            proposed_n50_page,

        "candidate_evidence":
            json.dumps(
                {
                    key: sorted(set(values))
                    for key, values
                    in sorted(candidate_evidence.items())
                },
                sort_keys=True,
            ),

        "notebook50_coordinate_rows":
            coordinate_rows,

        "notebook50_maximum_y":
            coordinate_max_y,

        "notebook50_maximum_x":
            coordinate_max_x,

        "coordinate_canvas_ok":
            coordinate_canvas_ok,

        "historical_candidate_paths":
            pipe_join(historical_candidate_paths),

        "observed_matches_historical_sha256":
            historical_candidate_hash_match,

        "observed_matches_historical_shape":
            historical_candidate_shape_match,

        "diagnostic_status":
            diagnostic_status,

        "authorised_for_decision_reuse":
            bool(
                diagnostic_status
                == "PROVEN_PIXEL_IDENTICAL_HISTORICAL_IMAGE"
            ),
    })


diagnostic = pd.DataFrame(diagnostic_rows)

require(
    len(diagnostic) == len(roster),
    "Diagnostic row count does not equal runtime roster count."
)

diagnostic.to_csv(
    D06 / "page_identity_diagnostic.csv",
    index=False,
)


# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------

summary = (
    diagnostic.groupby(
        "diagnostic_status",
        dropna=False,
    )
    .size()
    .reset_index(name="pages")
    .sort_values(
        ["pages", "diagnostic_status"],
        ascending=[False, True],
    )
)

summary.to_csv(
    D06 / "page_identity_summary.csv",
    index=False,
)

authorised = diagnostic[
    diagnostic["authorised_for_decision_reuse"]
].copy()

alias_but_unproven = diagnostic[
    diagnostic["proposed_notebook50_page"].notna()
    & ~diagnostic["authorised_for_decision_reuse"]
].copy()

no_alias = diagnostic[
    diagnostic["proposed_notebook50_page"].isna()
].copy()

authorised.to_csv(
    D06 / "proven_registered_replay_pages.csv",
    index=False,
)

alias_but_unproven.to_csv(
    D06 / "alias_found_but_not_proven.csv",
    index=False,
)

no_alias.to_csv(
    D06 / "no_notebook50_identity_match.csv",
    index=False,
)


# ------------------------------------------------------------
# Machine-readable verdict
# ------------------------------------------------------------

verdict = {
    "runtime_pages": int(len(diagnostic)),
    "notebook50_pages": int(len(n50_pages)),
    "exact_runtime_page_name_matches": int(
        diagnostic["candidate_evidence"]
        .str.contains(
            "EXACT_RUNTIME_PAGE",
            regex=False,
            na=False,
        )
        .sum()
    ),
    "pages_with_one_strong_candidate": int(
        diagnostic[
            "strong_candidate_notebook50_pages"
        ]
        .fillna("")
        .map(
            lambda value:
                len([
                    item
                    for item in value.split("|")
                    if item
                ]) == 1
        )
        .sum()
    ),
    "pixel_identical_registered_pages": int(
        diagnostic[
            "authorised_for_decision_reuse"
        ].sum()
    ),
    "alias_found_but_not_proven": int(
        len(alias_but_unproven)
    ),
    "no_resolved_alias": int(
        len(no_alias)
    ),
    "page_pair_resolution_files_found": int(
        len(resolution_paths)
    ),
    "historical_notebook50_images_found": int(
        len(historical_images)
    ),
    "decision": (
        "REGISTERED_DECISION_REUSE_AVAILABLE_FOR_PROVEN_PAGES"
        if len(authorised)
        else
        "NO_NOTEBOOK50_DECISION_REUSE_AUTHORISED"
    ),
    "important": (
        "A matching alias or compatible canvas does not prove that "
        "Notebook 50 y/x decisions belong to the current observed "
        "image. Only exact historical image identity is currently "
        "marked authorised."
    ),
}

atomic_json(
    D06 / "identity_diagnostic.json",
    verdict,
)


# ------------------------------------------------------------
# Console report
# ------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 50 / NOTEBOOK 72 PAGE-IDENTITY DIAGNOSTIC")
print("=" * 80)

print("Runtime roster pages:", len(diagnostic))
print("Notebook 50 page IDs:", len(n50_pages))
print(
    "Existing page_pair_resolution.csv files:",
    len(resolution_paths),
)
print(
    "Historical Notebook 50 image candidates:",
    len(historical_images),
)

print("\nDiagnostic status:")
print(summary.to_string(index=False))

print("\nVerdict:")
print(json.dumps(verdict, indent=2))

print("\nFirst Notebook 50 page IDs:")
print(n50_pages[:30])

print("\nFirst runtime page IDs:")
print(
    diagnostic["runtime_page"]
    .head(30)
    .tolist()
)

print("\nExamples with a proposed historical alias:")

example_columns = [
    "runtime_page",
    "proposed_notebook50_page",
    "diagnostic_status",
    "coordinate_canvas_ok",
    "observed_matches_historical_sha256",
    "observed_matches_historical_shape",
]

display(
    diagnostic[
        diagnostic[
            "proposed_notebook50_page"
        ].notna()
    ][example_columns]
    .head(30)
)

print("\nOutput files:")
print(" ", D06 / "page_identity_diagnostic.csv")
print(" ", D06 / "page_identity_summary.csv")
print(" ", D06 / "notebook50_page_inventory.csv")
print(" ", D06 / "page_pair_resolution_candidates.csv")
print(" ", D06 / "historical_notebook50_image_candidates.csv")
print(" ", D06 / "proven_registered_replay_pages.csv")
print(" ", D06 / "alias_found_but_not_proven.csv")
print(" ", D06 / "no_notebook50_identity_match.csv")
print(" ", D06 / "identity_diagnostic.json")


# ------------------------------------------------------------
# Deliberately stop before page processing
# ------------------------------------------------------------

if len(authorised) == 0:
    raise RuntimeError(
        "Diagnostic completed, but no runtime page has been proven "
        "pixel-identical to a registered Notebook 50 historical "
        "image. Do not reuse Notebook 50 coordinate decisions. "
        f"Inspect {D06 / 'page_identity_summary.csv'} and "
        f"{D06 / 'page_identity_diagnostic.csv'}."
    )

print()
print(
    f"{len(authorised)} page(s) have exact historical image identity "
    "and may be eligible for registered decision replay."
)

if len(authorised) < len(diagnostic):
    raise RuntimeError(
        f"Only {len(authorised)} of {len(diagnostic)} runtime pages "
        "have proven Notebook 50 identity. The remaining pages need "
        "the frozen deployable scorer; they must not inherit decisions "
        "by filename guess or page ordering."
    )

In [ ]:
# ============================================================
# 6. Run frozen V8.8 operator on eligible reviewed pages
# ============================================================

processed = []
failures = []

require(
    len(roster) > 0,
    "Runtime roster is empty."
)

require(
    hasattr(v88_module, "run_page"),
    "Frozen V8.8 adapter must expose run_page()."
)

print("Runtime pages:", len(roster))
print("Operator:", getattr(v88_module, "OPERATOR_ID", None))
print("Operator version:", getattr(v88_module, "OPERATOR_VERSION", None))
print()


for _, row in roster.iterrows():

    page = str(row["page"])

    print(f"Processing {page} ...")

    try:
        # ----------------------------------------------------
        # Concrete reviewed runtime inputs only
        # ----------------------------------------------------
        observed_path = Path(str(row["observed_path"]))
        seed_mask_path = Path(str(row["seed_mask_path"]))

        require(
            observed_path.is_file(),
            f"{page}: observed page missing: {observed_path}"
        )

        require(
            seed_mask_path.is_file(),
            f"{page}: authoritative seed mask missing: {seed_mask_path}"
        )

        observed = imread(observed_path)
        seed = imread(seed_mask_path)

        require(
            observed.ndim == 2,
            f"{page}: observed page is not 2-D: {observed.shape}"
        )

        require(
            seed.ndim == 2,
            f"{page}: seed mask is not 2-D: {seed.shape}"
        )

        require(
            observed.shape == seed.shape,
            f"{page}: observed/mask shape mismatch: "
            f"{observed.shape} vs {seed.shape}"
        )

        # ----------------------------------------------------
        # Bind the page identity to the frozen operator.
        #
        # The Notebook 50 registered V8.8 decision table is
        # keyed by page/y/x, so page identity is required.
        # ----------------------------------------------------
        page_operator = operator.for_page(page)

        # ----------------------------------------------------
        # Execute clean-free V8.8 page operation
        # ----------------------------------------------------
        result = v88_module.run_page(
            observed,
            seed,
            page_operator,
        )

        # ----------------------------------------------------
        # Basic result contract
        # ----------------------------------------------------
        require(
            result.corrected.shape == observed.shape,
            f"{page}: corrected-page shape mismatch"
        )

        require(
            result.corrected.dtype == observed.dtype,
            f"{page}: corrected-page dtype mismatch: "
            f"{result.corrected.dtype} vs {observed.dtype}"
        )

        require(
            result.baseline.shape == observed.shape,
            f"{page}: baseline shape mismatch"
        )

        require(
            result.prepared_mask.shape == observed.shape,
            f"{page}: prepared-mask shape mismatch"
        )

        # ----------------------------------------------------
        # Authoritative-boundary immutability
        # ----------------------------------------------------
        outside = result.prepared_mask == 0

        outside_drift = int(
            np.sum(
                result.corrected[outside]
                != observed[outside]
            )
        )

        require(
            outside_drift == 0,
            f"{page}: {outside_drift} changed pixels outside "
            "the authoritative prepared mask"
        )

        # ----------------------------------------------------
        # V8.8 fallback identity
        #
        # Everywhere except SAFE_REMOVE, the result must
        # reproduce the frozen fallback exactly.
        # ----------------------------------------------------
        safe_remove = np.asarray(result.safe_remove).astype(bool)

        require(
            safe_remove.shape == observed.shape,
            f"{page}: SAFE_REMOVE shape mismatch"
        )

        non_safe_regression = int(
            np.sum(
                result.corrected[~safe_remove]
                != result.baseline[~safe_remove]
            )
        )

        require(
            non_safe_regression == 0,
            f"{page}: {non_safe_regression} pixels differ from "
            "the frozen fallback outside SAFE_REMOVE"
        )

        # ----------------------------------------------------
        # Durable page outputs
        # ----------------------------------------------------
        page_dir = D06 / page
        page_dir.mkdir(parents=True, exist_ok=False)

        corrected_path = page_dir / "corrected.png"
        baseline_path = page_dir / "baseline.png"
        prepared_mask_path = page_dir / "prepared_mask.png"
        safe_remove_path = page_dir / "safe_remove.png"
        m5_remove_path = page_dir / "m5_remove.png"

        require(
            cv2.imwrite(
                str(corrected_path),
                result.corrected
            ),
            f"{page}: failed to write corrected page"
        )

        require(
            cv2.imwrite(
                str(baseline_path),
                result.baseline
            ),
            f"{page}: failed to write baseline page"
        )

        require(
            cv2.imwrite(
                str(prepared_mask_path),
                result.prepared_mask.astype(np.uint8) * 255
            ),
            f"{page}: failed to write prepared mask"
        )

        require(
            cv2.imwrite(
                str(safe_remove_path),
                safe_remove.astype(np.uint8) * 255
            ),
            f"{page}: failed to write SAFE_REMOVE"
        )

        require(
            cv2.imwrite(
                str(m5_remove_path),
                np.asarray(result.m5_remove).astype(np.uint8) * 255
            ),
            f"{page}: failed to write M5_REMOVE"
        )

        audit_path = page_dir / "audit.json"

        atomic_json(
            audit_path,
            result.audit
        )

        processed.append({
            "page": page,
            "cohort": row["cohort"],
            "status": "PROCESSED",
            "observed_path": str(observed_path),
            "seed_mask_path": str(seed_mask_path),
            "corrected_path": str(corrected_path),
            "baseline_path": str(baseline_path),
            "audit_path": str(audit_path),

            "prepared_mask_pixels":
                int(np.asarray(result.prepared_mask).astype(bool).sum()),

            "safe_remove_pixels":
                int(safe_remove.sum()),

            "m5_remove_pixels":
                int(np.asarray(result.m5_remove).astype(bool).sum()),

            "changed_pixels":
                int(np.sum(result.corrected != observed)),

            "outside_mask_drift":
                outside_drift,

            "non_safe_fallback_regression":
                non_safe_regression,

            "corrected_sha256":
                sha256_file(corrected_path),

            "baseline_sha256":
                sha256_file(baseline_path),
        })

        print(
            f"  PASS | "
            f"M3={processed[-1]['prepared_mask_pixels']:,} | "
            f"SAFE_REMOVE={processed[-1]['safe_remove_pixels']:,} | "
            f"M5_REMOVE={processed[-1]['m5_remove_pixels']:,} | "
            f"changed={processed[-1]['changed_pixels']:,}"
        )

    except Exception as exc:

        failure = {
            "page": page,
            "cohort": row.get("cohort"),
            "observed_path": str(row.get("observed_path", "")),
            "seed_mask_path": str(row.get("seed_mask_path", "")),
            "error_type": type(exc).__name__,
            "error": str(exc),
        }

        failures.append(failure)

        print(
            f"  FAIL | {type(exc).__name__}: {exc}"
        )


# ============================================================
# Write results BEFORE deciding whether the run can continue
# ============================================================

processed_columns = [
    "page",
    "cohort",
    "status",
    "observed_path",
    "seed_mask_path",
    "corrected_path",
    "baseline_path",
    "audit_path",
    "prepared_mask_pixels",
    "safe_remove_pixels",
    "m5_remove_pixels",
    "changed_pixels",
    "outside_mask_drift",
    "non_safe_fallback_regression",
    "corrected_sha256",
    "baseline_sha256",
]

failure_columns = [
    "page",
    "cohort",
    "observed_path",
    "seed_mask_path",
    "error_type",
    "error",
]

processed_df = pd.DataFrame(
    processed,
    columns=processed_columns,
)

failure_df = pd.DataFrame(
    failures,
    columns=failure_columns,
)

processed_df.to_csv(
    D06 / "processed_pages.csv",
    index=False,
)

failure_df.to_csv(
    D06 / "processing_failures.csv",
    index=False,
)


# ============================================================
# Console diagnostic
# ============================================================

print()
print("=" * 80)
print("V8.8 PAGE PROCESSING SUMMARY")
print("=" * 80)
print("Runtime roster:", len(roster))
print("Processed successfully:", len(processed_df))
print("Failed:", len(failure_df))

if len(failure_df):

    print()
    print("Failure types:")
    print(
        failure_df["error_type"]
        .value_counts()
        .to_string()
    )

    print()
    print("First failures:")

    for _, failure in failure_df.head(10).iterrows():
        print(
            f"  {failure['page']}: "
            f"{failure['error_type']}: "
            f"{failure['error']}"
        )


# ============================================================
# Fail closed
# ========================================================

require(
    len(processed_df) > 0,
    "No pages processed successfully. "
    f"See {D06 / 'processing_failures.csv'} for the real per-page errors."
)

require(
    len(failure_df) == 0,
    f"{len(failure_df)} runtime page(s) failed V8.8 processing. "
    f"See {D06 / 'processing_failures.csv'}."
)

print()
print("All runtime pages processed successfully.")

## 7. Prediction/output lock

In [ ]:
output_rows=[]
for page in page_integrity.page.astype(str):
    d=D03/page
    output_rows.append({'page':page,'fallback_sha256':sha256_file(d/'fallback_corrected.png'),'corrected_sha256':sha256_file(d/'v8_8_corrected.png'),
                        'decisions_semantic_sha256':semantic_npz_hash(d/'decision_map.npz'),'probabilities_semantic_sha256':semantic_npz_hash(d/'probabilities.npz'),
                        'changed_sha256':sha256_file(d/'changed_pixels.png'),'page_audit_sha256':sha256_file(d/'page_audit.json')})
outputs=pd.DataFrame(output_rows).sort_values('page'); outputs.to_csv(D05/'page_output_hashes.csv',index=False)
inputs=page_integrity[['page','cohort','observed_sha256','seed_mask_sha256','prepared_mask_sha256']].sort_values('page'); inputs.to_csv(D05/'input_hashes.csv',index=False)
access=pd.read_csv(D04/'runtime_access_audit.csv'); clean_pre=int(((access.clean_path==True)&(access.lock_state!='CLOSED')).sum()) if len(access) else 0
lock_payload={'schema':'notebook72-prediction-lock-v1','runtime_roster_sha256':sha256_file(D02/'runtime_roster.csv'),'input_hashes_sha256':sha256_file(D05/'input_hashes.csv'),
              'operator_identity_sha256':sha256_file(D01/'operator_identity.json'),'page_output_hashes_sha256':sha256_file(D05/'page_output_hashes.csv'),
              'runtime_access_audit_sha256':sha256_file(D04/'runtime_access_audit.csv'),'replay_test_sha256':sha256_file(D09/'replay_test.json'),
              'processed_pages':len(page_integrity),'failed_pages':len(failure_df),'clean_reads_before_lock':clean_pre}
require(clean_pre==0,'Prediction lock refused: clean was read before lock.')
lock_id=object_hash(lock_payload); lock={'lock_id':lock_id,'state':'CLOSED','created_utc':datetime.now(timezone.utc).isoformat(),'payload':lock_payload}
atomic_json(D05/'prediction_lock.json',lock)
LOCK_STATE='CLOSED'
print('PREDICTION LOCK CLOSED:',lock_id)

## 8. Post-lock paired truth evaluation

This cell is the first place that resolves `source/clean`. It verifies the current prediction lock before loading any clean page. Metrics preserve the project distinction between structural safety and cosmetic usefulness.

In [ ]:
current_lock=read_json(D05/'prediction_lock.json')
require(current_lock.get('state')=='CLOSED' and current_lock.get('lock_id')==lock_id,'Prediction lock is not the current CLOSED lock.')
require(current_lock['payload']['page_output_hashes_sha256']==sha256_file(D05/'page_output_hashes.csv'),'Page outputs changed after lock.')

CLEAN_DIR=Path(CLEAN_DIR_LITERAL)  # first permitted resolution
clean_map=image_map(CLEAN_DIR)

def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if not ref.sum(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1); recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum(); return precision,recall,2*precision*recall/max(precision+recall,1e-12)

eval_rows=[]
for row in page_integrity.itertuples(index=False):
    page=str(row.page)
    if row.cohort!='B_PAIRED_AS_ANNOTATED_ONLY' or page not in clean_map: continue
    d=D03/page; clean=imread(clean_map[page]); observed=imread(d/'observed.png'); corrected=imread(d/'v8_8_corrected.png'); mask=imread(d/'prepared_authoritative_mask.png')>0
    if not (clean.shape==observed.shape==corrected.shape==mask.shape):
        eval_rows.append({'page':page,'scored':False,'reason':'POST_LOCK_SHAPE_MISMATCH'}); continue
    ci,oi,ri=printed_ink(clean),printed_ink(observed),printed_ink(corrected); ip,ir,if1=tolerant_scores(ci,ri,mask); ep,er,ef1=tolerant_scores(edges(clean),edges(corrected),mask)
    diff=np.abs(clean.astype(np.float32)-corrected.astype(np.float32)); removed_print=float(np.sum(ci&~ri&mask)/max(np.sum(ci&mask),1)); residual_annotation=float(np.sum(ri&~ci&mask)/max(np.sum(mask),1))
    eval_rows.append({'page':page,'cohort':row.cohort,'scored':True,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,
                      'inside_mask_mae':float(diff[mask].mean()) if mask.any() else 0.0,'whole_page_mae':float(diff.mean()),'removed_print_fraction':removed_print,'residual_annotation_fraction':residual_annotation,
                      'catastrophic_print_damage':bool(removed_print>.01 or (np.isfinite(ir) and ir<.95))})
paired=pd.DataFrame(eval_rows); paired.to_csv(D06/'paired_quantitative_results.csv',index=False)
print('Truth-scored paired pages:',int(paired.scored.sum()) if len(paired) and 'scored' in paired else 0)

## 9. Generate blinded A/B review packages for every processed page

In [ ]:
rng=random.Random(RANDOM_SEED); review_keys=[]
for row in page_integrity.itertuples(index=False):
    page=str(row.page); d=D03/page; rd=D07/page; rd.mkdir(parents=True,exist_ok=False)
    original=imread(d/'observed.png'); corrected=imread(d/'v8_8_corrected.png'); corrected_is_a=bool(rng.getrandbits(1))
    A=corrected if corrected_is_a else original; B=original if corrected_is_a else corrected
    cv2.imwrite(str(rd/'review_A.png'),A); cv2.imwrite(str(rd/'review_B.png'),B)
    key={'page':page,'cohort':row.cohort,'A':'corrected' if corrected_is_a else 'original','B':'original' if corrected_is_a else 'corrected'}
    atomic_json(rd/'review_key.json',key); review_keys.append(key)
pd.DataFrame(review_keys).to_csv(D07/'sealed_review_keys.csv',index=False)
review_template=pd.DataFrame([{'page':r['page'],'reviewer_code':'','outcome':'','annotation_removal':'','notation_damage':'','preference':'','performability':'','rejection_region':'','reason':'','reviewed_utc':''} for r in review_keys])
review_template.to_csv(D07/'performance_usability_reviews.csv',index=False)
print('Review packages:',len(review_keys))

## 10. Blinded performance-usability reviewer

Allowed outcomes: `PERFORMANCE_READY`, `PERFORMANCE_READY_WITH_MINOR_RESIDUE`, `REVIEW_REQUIRED`, `UNUSABLE_ANNOTATION_REMAINS`, `UNUSABLE_PRINT_DAMAGE`.

In [ ]:
import gradio as gr
review_df=pd.read_csv(D07/'performance_usability_reviews.csv',dtype=str).fillna('')
pages=review_df.page.tolist(); first=pages[0]
OUTCOMES=['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE','REVIEW_REQUIRED','UNUSABLE_ANNOTATION_REMAINS','UNUSABLE_PRINT_DAMAGE']

def review_pair(page): return str(D07/page/'review_A.png'),str(D07/page/'review_B.png')
def save_review(page,reviewer,outcome,removal,damage,preference,performability,region,reason):
    global review_df
    require(outcome in OUTCOMES,'Select an outcome.'); require(bool(reviewer.strip()),'Reviewer code required.')
    i=review_df.index[review_df.page==page]; require(len(i)==1,'Review page mismatch.')
    review_df.loc[i[0],:]=[page,reviewer,outcome,removal,damage,preference,performability,region,reason,datetime.now(timezone.utc).isoformat()]
    review_df.to_csv(D07/'performance_usability_reviews.csv',index=False)
    return f'Saved {page}: {outcome}'

with gr.Blocks(title='Notebook 72 Blinded Performance-Usability Review') as app:
    gr.Markdown('# Notebook 72 blinded page review\nA and B are randomised. Judge whether the corrected page is clean and trustworthy enough to perform from. Do not open diagnostic artefacts before recording the initial judgement.')
    page=gr.Dropdown(pages,value=first,label='Page')
    with gr.Row(): A=gr.Image(value=str(D07/first/'review_A.png'),label='A',height=720); B=gr.Image(value=str(D07/first/'review_B.png'),label='B',height=720)
    reviewer=gr.Textbox(label='Reviewer code'); outcome=gr.Dropdown(OUTCOMES,label='Outcome')
    removal=gr.Radio(['complete','partial','none'],label='Annotation removal'); damage=gr.Radio(['none','uncertain','visible_material_damage'],label='Notation damage')
    preference=gr.Radio(['A_better','same','B_better'],label='Preference'); performability=gr.Radio(['yes','yes_with_minor_concern','no'],label='Would you perform from the preferred page?')
    region=gr.Textbox(label='Rejection/referral region'); reason=gr.Textbox(label='Reason/notes'); save=gr.Button('Lock this initial judgement',variant='primary'); status=gr.Textbox(interactive=False)
    page.change(review_pair,[page],[A,B]); save.click(save_review,[page,reviewer,outcome,removal,damage,preference,performability,region,reason],[status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 11. Decode reviews, calculate gates and write combined decision package

In [ ]:
reviews=pd.read_csv(D07/'performance_usability_reviews.csv',dtype=str).fillna('')
require((reviews.outcome!='').all(),'Review is incomplete. Record one outcome for every processed page.')
keys=pd.read_csv(D07/'sealed_review_keys.csv',dtype=str)
reviews=reviews.merge(keys,on=['page'],validate='one_to_one')
reviews['corrected_preferred']=((reviews.preference=='A_better')&(reviews.A=='corrected'))|((reviews.preference=='B_better')&(reviews.B=='corrected'))
reviews.to_csv(D07/'performance_usability_reviews_decoded.csv',index=False)
summary=reviews.groupby(['cohort','outcome'],as_index=False).size().rename(columns={'size':'pages'}); summary.to_csv(D07/'performance_usability_summary.csv',index=False)

paired_scored=paired[paired.get('scored',False)==True] if len(paired) else pd.DataFrame()
structural_ok=bool(len(paired_scored)==0 or (~paired_scored.catastrophic_print_damage.astype(bool)).all())
no_visible_damage=not reviews.outcome.eq('UNUSABLE_PRINT_DAMAGE').any()
ready=reviews.outcome.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE'])
useful_share=float(ready.mean())
source_summary=page_integrity[['page','cohort','status']].merge(reviews[['page','outcome','corrected_preferred']],on='page',how='left')
source_summary.groupby('cohort',as_index=False).agg(pages=('page','nunique'),performance_ready=('outcome',lambda x:int(x.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE']).sum())),unusable_print_damage=('outcome',lambda x:int((x=='UNUSABLE_PRINT_DAMAGE').sum())),corrected_preferred=('corrected_preferred','sum')).to_csv(D08/'results_by_source_group.csv',index=False)
source_summary.to_csv(D08/'results_by_page.csv',index=False)

gates={
 'G1_frozen_identity':replay.get('passed') is True,
 'G2_roster_completeness':len(page_integrity)+len(failure_df)==len(roster),
 'G3_truth_firewall':clean_pre==0,
 'G4_outside_mask_immutability':bool((page_integrity.outside_mask_drift==0).all()),
 'G5_fallback_equivalence':bool((page_integrity.nonintervention_fallback_regression==0).all()),
 'G6_replay':replay.get('passed') is True,
 'G7_structural_safety':bool(structural_ok and no_visible_damage),
 'G8_cosmetic_usefulness':bool(useful_share>0),
 'G9_source_stability':'REVIEW_REQUIRED_HUMAN_INTERPRETATION',
 'G10_auditability':bool(outputs.page.nunique()==len(page_integrity) and (page_integrity.changed_unjustified==0).all())
}
required_bool=['G1_frozen_identity','G2_roster_completeness','G3_truth_firewall','G4_outside_mask_immutability','G5_fallback_equivalence','G6_replay','G7_structural_safety','G8_cosmetic_usefulness','G10_auditability']
first_failed=next((k for k in required_bool if gates[k] is not True),None)
verdict='PASS_SUPERVISED_OPERATIONAL_TRIAL_EVIDENCE_READY_FOR_DECISION' if first_failed is None else 'FAIL_NOTEBOOK72_OPERATIONAL_TRIAL'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'lock_id':lock_id,'eligible_pages':len(roster),'processed_pages':len(page_integrity),'failed_pages':len(failure_df),'excluded_pages':len(exclusions),'paired_truth_scored':len(paired_scored),'performance_ready_share':useful_share,'promotion_authority':False,'automatic_output_authority':False,'mask_authority_automation':False}
atomic_json(OUT/'notebook72_verdict.json',record)
atomic_json(OUT/'experiment_manifest.json',{'run_id':RUN_ID,'operator_identity':operator_identity,'prediction_lock':lock,'replay':replay,'record':record})
lines=['NOTEBOOK 72 / FROZEN V8.8 ANNOTATED-ONLY SUPERVISED OPERATIONAL TRIAL','',f'Verdict: {verdict}',f'Prediction lock: {lock_id}',f'Eligible / processed / failed / excluded: {len(roster)} / {len(page_integrity)} / {len(failure_df)} / {len(exclusions)}',f'Paired truth-scored pages: {len(paired_scored)}',f'Performance-ready share: {useful_share:.3%}','','GATES']+[f'- {k}: {v}' for k,v in gates.items()]+['','No retraining, recalibration, threshold movement, automatic mask authority or production release was performed.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines))

## 12. Standard Final Upload Package

In [ ]:
for sub in ['tables','review','visuals','audit']: (UPLOAD/sub).mkdir(parents=True,exist_ok=True)
curated=[OUT/'executive_summary.txt',OUT/'notebook72_verdict.json',OUT/'experiment_manifest.json',D05/'prediction_lock.json',D02/'runtime_roster.csv',D02/'exclusions.csv',D04/'page_integrity.csv',D04/'runtime_access_audit.csv',D05/'page_output_hashes.csv',D06/'paired_quantitative_results.csv',D07/'performance_usability_reviews_decoded.csv',D07/'performance_usability_summary.csv',D08/'results_by_page.csv',D08/'results_by_source_group.csv',D08/'failure_and_exception_register.csv',D09/'replay_test.json',D01/'operator_artifact_inventory.csv']
rows=[]
for src in curated:
    require(src.exists(),f'Missing required upload artefact: {src}')
    group='review' if 'performance_usability' in src.name else ('audit' if src.suffix=='.json' or 'audit' in src.name or 'integrity' in src.name or 'hash' in src.name else 'tables')
    dst=UPLOAD/group/src.name; shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
# Include low-resolution A/B review images, but keep full runtime pages in the run directory.
for page in page_integrity.page.astype(str):
    for name in ['review_A.png','review_B.png']:
        src=D07/page/name; dst=UPLOAD/'review'/f'{page}_{name}'; shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
manifest=pd.DataFrame(rows).sort_values('relative_path'); manifest.to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(manifest.relative_path))
(UPLOAD/'README.txt').write_text('Notebook 72 review package. Full-resolution runtime outputs remain under 03_runtime_outputs and are indexed by page_output_hashes.csv. Clean truth was unavailable to runtime code before the CLOSED prediction lock. This package has no production-release authority.\n')
zip_path=UPLOAD/'notebook72_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print('Upload bundle:',zip_path,'SHA-256:',sha256_file(zip_path))

## Outputs to inspect first
- `executive_summary.txt`
- `notebook72_verdict.json`
- `05_prediction_lock/prediction_lock.json`
- `04_runtime_integrity/page_integrity.csv`
- `06_paired_truth_evaluation/paired_quantitative_results.csv`
- `07_blinded_review/performance_usability_summary.csv`
- `08_combined_results/failure_and_exception_register.csv`
- `upload/notebook72_upload_bundle.zip`

If operator resolution, replay, truth firewall, immutability or fallback equivalence fails, do not suppress the error. Fix the frozen artefact or lineage issue and create a new run.